# Paper 1: Comparative CNN Analysis for Skin Lesion Classification (HAM10000)
### Complete Pipeline: Download -> Preprocess -> Train 6 Models -> Compare Results

**IMPORTANT BEFORE STARTING:**
1. Runtime -> Change runtime type -> T4 GPU -> Save
2. Have your `kaggle.json` file ready (from kaggle.com -> Settings -> Create New API Token)

## STEP 1: Upload Kaggle API Key & Download Dataset

In [ ]:
# This will open a file upload button - select your kaggle.json file
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

In [ ]:
# Set up Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download HAM10000 dataset (~2.6 GB, takes 2-5 minutes)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

# Unzip it
!mkdir -p /content/ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/ham10000
print("\nDataset downloaded and extracted successfully!")

In [ ]:
# Verify what got downloaded
import os
for f in os.listdir('/content/ham10000'):
    print(f)

## STEP 2: Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, VGG16, DenseNet121, EfficientNetB0, MobileNetV2, InceptionV3
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

## STEP 3: Load Metadata & Build Image Path Dataframe

In [ ]:
# Load metadata CSV (contains image_id -> lesion type mapping)
metadata = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
print("Metadata shape:", metadata.shape)
metadata.head()

In [ ]:
# Images are split across two folders in this Kaggle version - combine paths
import glob
all_image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
image_path_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in all_image_paths}
print("Total images found:", len(image_path_dict))

metadata['image_path'] = metadata['image_id'].map(image_path_dict.get)
metadata = metadata.dropna(subset=['image_path']).reset_index(drop=True)
print("Metadata rows with valid image path:", len(metadata))

## STEP 4: Explore Class Distribution (Confirm Imbalance)

In [ ]:
class_counts = metadata['dx'].value_counts()
print(class_counts)

plt.figure(figsize=(8,5))
class_counts.plot(kind='bar', color='coral')
plt.title('Class Distribution in HAM10000')
plt.xlabel('Lesion Type')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

# Label meaning for reference
label_names = {
    'akiec': 'Actinic Keratoses', 'bcc': 'Basal Cell Carcinoma', 'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma', 'mel': 'Melanoma', 'nv': 'Melanocytic Nevi', 'vasc': 'Vascular Lesion'
}
for k, v in label_names.items():
    print(f"{k}: {v}")

## STEP 5: Train / Validation / Test Split (Stratified)

In [ ]:
classes = sorted(metadata['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
metadata['label_idx'] = metadata['dx'].map(class_to_idx)

train_df, temp_df = train_test_split(metadata, test_size=0.3, stratify=metadata['dx'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['dx'], random_state=42)

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)
print("\nTrain class distribution:\n", train_df['dx'].value_counts())

## STEP 6: Image Data Generators (Loading + Augmentation)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def make_generators(preprocess_func):
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_func,
        rotation_range=45,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2]
    )
    val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_func)

    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=True, seed=42
    )
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=False
    )
    test_gen = val_test_datagen.flow_from_dataframe(
        test_df, x_col='image_path', y_col='dx', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=classes, shuffle=False
    )
    return train_gen, val_gen, test_gen

print("Generator function ready. Each model uses its own preprocessing function.")

## STEP 7: Define the 6 Models to Compare

In [ ]:
MODEL_CONFIGS = {
    'ResNet50': (ResNet50, resnet_preprocess),
    'VGG16': (VGG16, vgg_preprocess),
    'DenseNet121': (DenseNet121, densenet_preprocess),
    'EfficientNetB0': (EfficientNetB0, efficientnet_preprocess),
    'MobileNetV2': (MobileNetV2, mobilenet_preprocess),
    'InceptionV3': (InceptionV3, inception_preprocess)
}

def build_model(base_model_fn, num_classes=7):
    base = base_model_fn(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False  # freeze base layers (transfer learning)

    inputs = tf.keras.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("6 models configured:", list(MODEL_CONFIGS.keys()))

## STEP 8: Train All 6 Models (This Is the Longest Step -- ~15-25 min per model)

Each model trains separately. Results are stored in `results_dict` for the final comparison table.

In [ ]:
results_dict = {}
trained_models = {}
EPOCHS = 20

for model_name, (model_fn, preprocess_func) in MODEL_CONFIGS.items():
    print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")

    train_gen, val_gen, test_gen = make_generators(preprocess_func)
    model = build_model(model_fn)

    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    # Evaluate on test set
    test_gen.reset()
    y_pred_probs = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_gen.classes

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    y_true_bin = label_binarize(y_true, classes=list(range(len(classes))))
    try:
        auc = roc_auc_score(y_true_bin, y_pred_probs, average='macro', multi_class='ovr')
    except Exception:
        auc = np.nan

    results_dict[model_name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1, 'AUC': auc,
        'y_true': y_true, 'y_pred': y_pred
    }
    trained_models[model_name] = model

    print(f"{model_name} -> Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    model.save(f'/content/{model_name}_ham10000.keras')

print("\n\nAll models trained and saved!")

## STEP 9: Final Comparison Table (Core Result for Your Paper)

In [ ]:
comparison_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['y_true', 'y_pred']}
    for name, res in results_dict.items()
}).T

comparison_df = comparison_df.sort_values('Accuracy', ascending=False)
print(comparison_df.round(4))
comparison_df.to_csv('/content/model_comparison_results.csv')

In [ ]:
# Bar chart comparing all models
fig, ax = plt.subplots(figsize=(10,6))
comparison_df[['Accuracy','Precision','Recall','F1-Score']].plot(kind='bar', ax=ax)
plt.title('CNN Architecture Comparison on HAM10000')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('/content/model_comparison_chart.png', dpi=300)
plt.show()

## STEP 10: Confusion Matrix for the Best Model

In [ ]:
best_model_name = comparison_df.index[0]
print("Best model:", best_model_name)

y_true = results_dict[best_model_name]['y_true']
y_pred = results_dict[best_model_name]['y_pred']

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/confusion_matrix_best_model.png', dpi=300)
plt.show()

print("\nDetailed classification report:\n")
print(classification_report(y_true, y_pred, target_names=classes))

## STEP 11: Download Your Results (Save to Your Computer)

In [ ]:
from google.colab import files
files.download('/content/model_comparison_results.csv')
files.download('/content/model_comparison_chart.png')
files.download('/content/confusion_matrix_best_model.png')

---
## What You Now Have for Paper 1
- Comparison table (Accuracy, Precision, Recall, F1, AUC) for 6 CNN architectures
- Bar chart comparing all models
- Confusion matrix + classification report for the best model
- Saved `.keras` model files for each architecture

**Next step:** Use these results to write the Results/Discussion section of Paper 1, following IEEE format.